# Mental Health Digital Shadows (MHDS) — Data Generation

This notebook generates the raw JSON records used to build the **Mental Health
Digital Shadows (MHDS)** dataset described in:

> *Digital shadows in mental health map how LLMs simulate depression, anxiety, and stress through language and psychometrics.*
>
> Franchino, E., Rizzi, R., De Duro, E. S., & Stella, M. 


For each run the script:

1. Randomly assigns a generation **mode** (`human` shadow or `llm` assistant) using a 75/25 split.
2. If `human`, samples a sociodemographic + psychological persona from the randomisation pool.
3. Builds a structured prompt asking the model to (a) reply to mental-health topic questions,
   (b) emit 10 emotional-recall words, (c) score and justify the 21 DASS-21 items.
4. Calls the model through any OpenAI-compatible endpoint (LM Studio, Ollama, vLLM, xAI API, ...).
5. Validates the JSON output against the expected schema and persists the full artifact.

Downstream parsing of the JSON dumps into the cleaned per-model CSVs distributed in the
[MHDS GitHub repository](https://github.com/MassimoStel/MHDS) is handled by a separate
script (Step 5–6 of Figure 1B in the paper) and is **not** part of this notebook.

**License:** CC0 1.0 Universal (public-domain dedication), as for the dataset itself.

## Imports

In [ ]:
from __future__ import annotations

import json
import logging
import os
import random
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional

from openai import OpenAI


## Configuration

To run a different model, change `DEFAULT_MODEL`: output files will automatically be sharded under `runs/<model_slug>/`.


In [ ]:
# ---- Configuration ----
# Override via environment variables when needed.
DEFAULT_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://127.0.0.1:1234/v1")
DEFAULT_MODEL    = os.getenv("LMSTUDIO_MODEL", "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B")
DEFAULT_API_KEY  = os.getenv("LMSTUDIO_API_KEY", "not-needed")  # LM Studio ignores the key

# ---- Sampling configuration ----
TEMPERATURE = 0.0       
MAX_TOKENS  = 2000      

# ---- Run configuration ----
N_RUNS         = 5000      
HUMAN_MODE_RATIO = 0.75    
OUTPUT_ROOT    = "MHDS"     


## Logging

A lightweight logger writes both to stdout and to a per-model log file so that
silently-skipped failures (caught by the main loop) leave an auditable trail.


In [ ]:
def _model_slug(model_name: str) -> str:
    """Filesystem-safe identifier derived from the model name."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", model_name).strip("_")


def _setup_logger(model_name: str, output_root: str) -> logging.Logger:
    """Configure a logger that writes to runs/<model_slug>/run.log and to stdout."""
    slug = _model_slug(model_name)
    log_dir = Path(output_root).expanduser().resolve() / slug
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / "run.log"

    logger = logging.getLogger(f"mhds.{slug}")
    logger.setLevel(logging.INFO)
    # avoid duplicate handlers when the cell is re-executed in a notebook
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s",
                            datefmt="%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(sh)

    logger.propagate = False
    return logger


logger = _setup_logger(DEFAULT_MODEL, OUTPUT_ROOT)
logger.info("Logger initialised for model=%s", DEFAULT_MODEL)


# Persona randomisation pool

Static lookup tables for the persona randomisation pool (Figure 1A in the paper):
cities, education levels, occupations, hobbies, and DASS severity descriptors.


In [ ]:
# ---- Cities ----
CITIES_ITA = ["Rome", "Milan", "Naples", "Bari", "Lecce", "Bologna"]

CITIES_USA = [
    "New York",
    "Los Angeles",
    "Kansas City",
    "Philadelphia",
    "Chicago",
    "Washington DC",
]

CITIES = CITIES_ITA + CITIES_USA

# ---- Education ----
EDU_LEVELS = [
    "no formal education",
    "primary school",
    "lower secondary",
    "upper secondary",
    "vocational diploma",
    "bachelor's degree",
    "master's degree",
    "PhD",
]

# ---- Occupations ----
SEMI_REGULATED_OCCUPATIONS = ["architect", "lawyer", "doctor", "pharmacist"]

OCCUPATIONS_UNI = [
    "software developer",
    "data scientist",
    "startup founder",
    "journalist",
    "translator",
    "UX designer",
    "accountant",
    "school teacher",
    "engineer",
    "nurse",
]

OCCUPATIONS_NON_UNI = [
    "barista",
    "sales associate",
    "electrician",
    "fitness trainer",
    "civil servant",
    "plumber",
    "carpenter",
    "mechanic",
    "retail worker",
    "delivery driver",
    "cleaner",
    "waiter/waitress",
    "hairdresser",
    "chef",
]

# ---- Hobbies ----
HOBBIES = [
    "hiking",
    "reading novels",
    "baking",
    "football",
    "running",
    "yoga",
    "board games",
    "photography",
    "gardening",
    "volunteering",
    "cinema",
    "classical music",
    "painting",
    "cycling",
    "videogames",
    "travel",
    "cooking",
    "birdwatching",
    "knitting",
    "rock climbing",
]

# ---- Psychological severity descriptors ----
SEVERITY_LEVELS = ["severe", "moderate", "light", "without"]

PSYCHOLOGY = {
    condition: [
        (
            f"with {level} {condition} symptoms"
            if level != "without"
            else f"without {condition} symptoms"
        )
        for level in SEVERITY_LEVELS
    ]
    for condition in ["depression", "anxiety", "stress"]
}

# ---- Generation modes ----
ROLE_MODES = ["human", "llm"]


## Psychological dimensions

- **OCEAN**: each Big Five trait gets a 0-100 score plus a categorical descriptor
  (`low` / `moderate` / `high`).
- **DASS levels**: severity descriptor sampled jointly across depression, anxiety,
  and stress, with a rejection-sampling rule that prevents implausible mixes
  (a `severe` symptom on one subscale forces the other two to be at least `moderate`).


In [ ]:
def choose_ocean() -> Dict[str, Dict[str, Any]]:
    """Generate OCEAN trait scores (0-100) plus a categorical descriptor."""

    def bucket(score: int) -> str:
        if score < 33:
            return "low"
        if score > 66:
            return "high"
        return "moderate"

    traits = {}
    for key in [
        "openness",
        "conscientiousness",
        "extraversion",
        "agreeableness",
        "neuroticism",
    ]:
        score = random.randint(0, 100)
        traits[key] = {"score": score, "level": bucket(score)}
    return traits


def choose_dass_levels() -> dict:
    """Sample a (depression, anxiety, stress) triplet of severity descriptors.

    Rejection-sampling rule: if any subscale is sampled as `severe` (index 0),
    the other two must also be at most `moderate` (index <= 1); otherwise the
    triplet is rejected and a new one is drawn.
    """
    scales = {
        "depression": {"severe": 0.7, "moderate": 0.5, "light": 0.7, "without": 0.7},
        "anxiety":    {"severe": 0.7, "moderate": 0.5, "light": 0.7, "without": 0.7},
        "stress":     {"severe": 0.7, "moderate": 0.5, "light": 0.7, "without": 0.7},
    }

    SEVERITY_THRESHOLD = 1  # index <= 1 means severe or moderate

    while True:
        indices = {
            condition: random.choices(range(4), weights=list(w.values()), k=1)[0]
            for condition, w in scales.items()
        }

        has_severe = any(i == 0 for i in indices.values())
        if has_severe and not all(i <= SEVERITY_THRESHOLD for i in indices.values()):
            continue

        return {
            condition: PSYCHOLOGY[condition][idx]
            for condition, idx in indices.items()
        }


## Demographics

- Age (weighted age-band sampling)
- Gender
- Sexual orientation
- Migration status (conditional on city)


In [ ]:
def choose_age() -> int:
    ranges = {
        (18, 21): 1.25,
        (22, 26): 1.2,
        (27, 31): 1.2,
        (32, 36): 1.2,
        (37, 41): 1,
        (42, 46): 1,
        (47, 51): 1,
        (52, 56): 1,
        (57, 61): 1,
        (62, 66): 1,
        (67, 71): 1,
        (72, 76): 1,
        (77, 81): 1,
        (82, 86): 1,
        (87, 90): 0.8,
    }
    ((low, high),) = random.choices(
        list(ranges.keys()), weights=list(ranges.values()), k=1
    )
    return random.randint(low, high)


def choose_gender() -> str:
    options = {
        "man":         5,
        "woman":       5,
        "non-binary":  0.5,
        "genderqueer": 0.5,
        "agender":     0.5,
        "transgender": 0.5,
    }
    return random.choices(list(options.keys()), weights=list(options.values()), k=1)[0]


def choose_sexual_orientation() -> str:
    options = {
        "heterosexual": 8,
        "homosexual":   0.5,
        "bisexual":     0.5,
        "asexual":      0.3,
    }
    return random.choices(list(options.keys()), weights=list(options.values()), k=1)[0]


def choose_migration_status(city_of_living: str) -> str:
    """Migration status is conditional on the city of residence."""
    if city_of_living in CITIES_ITA:
        if random.randint(0, 100) < 21:
            return "immigrant"
        return "native-born Italian"
    else:
        if random.randint(0, 100) < 21:
            return "immigrant"
        return "native-born American"


## Socioeconomic

- Education
- Employment status (conditional on age and education)
- Occupation (conditional on education and employment)


In [ ]:
def choose_education(age: int) -> str:
    if age == 18:
        options = {
            "no formal education": 1,
            "primary school":      1,
            "lower secondary":     1,
            "upper secondary":     6,
            "vocational diploma":  1,
        }
    elif age <= 21:
        options = {
            "no formal education": 1,
            "primary school":      1,
            "upper secondary":     4,
            "vocational diploma":  2,
            "bachelor's degree": 2,
        }
    elif age <= 24:
        options = {
            "no formal education": 1,
            "primary school":      1,
            "upper secondary":     3,
            "vocational diploma":  1,
            "bachelor's degree": 3,
            "master's degree":   1,
        }
    else:
        options = {
            "no formal education": 1,
            "primary school":      2,
            "lower secondary":     5,
            "upper secondary":     10,
            "vocational diploma":  7,
            "bachelor's degree": 6,
            "master's degree":   3,
            "PhD":                 1,
        }

    return random.choices(list(options.keys()), weights=list(options.values()), k=1)[0]


def choose_employment_status(age: int, education_level: str) -> str:
    if age < 22:
        if education_level in {"no formal education", "primary school"}:
            options = {
                "employed full-time": 3,
                "employed part-time": 3,
                "part-time student":  0.5,
                "self-employed":      1.5,
                "unemployed":         2,
            }
        else:
            options = {
                "full-time student":  6,
                "employed full-time": 1,
                "employed part-time": 1,
                "self-employed":      1,
                "unemployed":         0.5,
                "part-time student":  0.5,
            }

    elif age < 30:
        if education_level in {"bachelor's degree", "master's degree", "PhD"}:
            options = {
                "full-time student":  4,
                "employed full-time": 3,
                "employed part-time": 1,
                "self-employed":      1,
                "unemployed":         1,
                "part-time student":  0.5,
            }
        else:
            options = {
                "employed full-time": 4,
                "employed part-time": 2,
                "self-employed":      1,
                "unemployed":         2,
            }

    elif age < 40:
        if education_level in {"master's degree", "PhD"}:
            options = {
                "employed full-time": 6,
                "employed part-time": 2,
                "self-employed":      1.5,
                "full-time student":  0.4,
                "unemployed":         1,
                "part-time student":  0.5,
            }
        else:
            options = {
                "employed full-time": 6,
                "employed part-time": 2,
                "self-employed":      2,
                "unemployed":         1,
            }

    elif age <= 65:
        if education_level in {
            "no formal education",
            "primary school",
            "lower secondary",
            "upper secondary",
        }:
            options = {
                "employed full-time": 4,
                "employed part-time": 2,
                "self-employed":      1,
                "unemployed":         3,
            }
        else:
            options = {
                "employed full-time": 6,
                "employed part-time": 2,
                "self-employed":      2,
                "full-time student":  0.2,
                "unemployed":         1,
            }

    elif age <= 75:
        options = {
            "retired":            8,
            "employed part-time": 1,
            "self-employed":      0.7,
            "unemployed":         0.3,
        }

    else:
        options = {
            "retired":            9.5,
            "employed part-time": 0.3,
            "self-employed":      0.1,
            "unemployed":         0.1,
        }

    return random.choices(list(options.keys()), weights=list(options.values()), k=1)[0]


def choose_occupation(education_level: str, employment_status: str) -> str:
    if employment_status in {"unemployed", "retired", "full-time student"}:
        return "not applicable"

    if education_level == "PhD":
        options = (
            [
                ("university researcher", 5),
                ("university professor",  4),
                ("scientist",             4),
            ]
            + [(job, 2)   for job in OCCUPATIONS_UNI]
            + [(job, 1)   for job in SEMI_REGULATED_OCCUPATIONS]
            + [(job, 0.3) for job in OCCUPATIONS_NON_UNI]
        )

    elif education_level == "master's degree":
        options = (
            [(job, 3)   for job in OCCUPATIONS_UNI]
            + [(job, 1.5) for job in SEMI_REGULATED_OCCUPATIONS]
            + [(job, 0.5) for job in OCCUPATIONS_NON_UNI]
        )

    elif education_level == "bachelor's degree":
        options = [(job, 3) for job in OCCUPATIONS_UNI] + [
            (job, 1) for job in OCCUPATIONS_NON_UNI
        ]

    else:
        return random.choice(OCCUPATIONS_NON_UNI)

    choices, weights = zip(*options)
    return random.choices(choices, weights=weights, k=1)[0]


## Family

- Marital status (conditional on age)
- Number of children (conditional on age, marital status, sexual orientation)


In [ ]:
def choose_marital_status(age: int) -> str:
    """Age-specific marital status distribution.

    Strongly reduces marriage at very young ages and divorce under 35.
    """
    if age < 22:
        options = {"single": 6, "in a relationship": 4, "married": 0.2}

    elif age < 25:
        options = {"single": 6, "in a relationship": 3, "married": 1}

    elif age < 35:
        options = {"single": 3, "in a relationship": 3, "married": 3, "divorced": 0.25}

    elif age < 60:
        options = {
            "single":            1.5,
            "in a relationship": 2,
            "married":           5,
            "divorced":          1.5,
            "widowed":           0.4,
        }

    elif age < 70:
        options = {
            "single":            1,
            "in a relationship": 1,
            "married":           4,
            "divorced":          2,
            "widowed":           1,
        }

    else:
        options = {
            "single":            1,
            "in a relationship": 1,
            "married":           4,
            "divorced":          2,
            "widowed":           2,
        }

    return random.choices(list(options.keys()), weights=list(options.values()), k=1)[0]


def choose_children(age: int, marital_status: str, sexual_orientation: str) -> int:
    """Number of children depends on age and marital status.

    For asexual individuals, having children remains possible but is somewhat
    less likely on average.
    """
    if marital_status == "single":
        if age < 22:
            options = {0: 20, 1: 1}
        elif age < 30:
            options = {0: 10, 1: 2, 2: 0.5}
        elif age < 40:
            options = {0: 5, 1: 4, 2: 2, 3: 0.5}
        else:
            options = {0: 4, 1: 4, 2: 3, 3: 1}

    elif marital_status == "in a relationship":
        if age < 22:
            options = {0: 15, 1: 0.5}
        elif age < 30:
            options = {0: 6, 1: 4, 2: 1}
        elif age < 40:
            options = {0: 2, 1: 4, 2: 4, 3: 2, 4: 0.5}
        else:
            options = {0: 2, 1: 3, 2: 4, 3: 2, 4: 1}

    elif marital_status == "married":
        if age < 22:
            options = {0: 8, 1: 0.5}
        elif age < 30:
            options = {0: 3, 1: 4, 2: 3, 3: 0.5}
        elif age < 40:
            options = {0: 1.5, 1: 3, 2: 4, 3: 2, 4: 1}
        else:
            options = {0: 1.5, 1: 3, 2: 4, 3: 2, 4: 1}

    else:  # divorced / widowed
        if age < 30:
            options = {0: 4, 1: 3, 2: 1}
        elif age < 40:
            options = {0: 2, 1: 4, 2: 4, 3: 2, 4: 1}
        else:
            options = {0: 1.5, 1: 3, 2: 4, 3: 2, 4: 1}

    if sexual_orientation == "asexual":
        options = {
            c: (w * 1.5 if c == 0 else w * 0.8 if c == 1 else w * 0.6)
            for c, w in options.items()
        }

    return random.choices(list(options.keys()), weights=list(options.values()), k=1)[0]


## Beliefs

- Religion


In [ ]:
def choose_religion() -> str:
    options = {
        "Christianity": 2,
        "Islam":        2,
        "Hinduism":     2,
        "Buddhism":     0.5,
        "Judaism":      0.5,
        "Atheist":      3,
        "Agnostic":     1,
    }
    return random.choices(list(options.keys()), weights=list(options.values()), k=1)[0]


## Persona dataclass and factory

In [ ]:
@dataclass
class Persona:
    age: int
    gender: str
    sexual_orientation: str
    occupation: str
    city_of_living: str
    employment_status: str
    education_level: str
    parents_education: Dict[str, str]
    marital_status: str
    children: int
    migration_status: str
    psychological_stress: str
    psychological_anxiety: str
    psychological_depression: str
    religious_beliefs: str
    hobbies: List[str]
    ocean: Dict[str, Dict[str, Any]]


def random_persona() -> Persona:
    # --- Psychological features ---
    psychological_feature = choose_dass_levels()

    # --- Demographics ---
    age = choose_age()
    gender = choose_gender()
    sexual_orientation = choose_sexual_orientation()
    city_of_living = random.choice(CITIES)
    migration_status = choose_migration_status(city_of_living)

    # --- Socioeconomic ---
    edu = choose_education(age)
    employment_status = choose_employment_status(age, edu)
    occupation = choose_occupation(edu, employment_status)
    marital_status = choose_marital_status(age)
    children = choose_children(age, marital_status, sexual_orientation)

    # --- Identity & beliefs ---
    religious_beliefs = choose_religion()

    return Persona(
        age=age,
        gender=gender,
        sexual_orientation=sexual_orientation,
        city_of_living=city_of_living,
        migration_status=migration_status,
        occupation=occupation,
        employment_status=employment_status,
        education_level=edu,
        parents_education={
            "parent_1": random.choice(EDU_LEVELS),
            "parent_2": random.choice(EDU_LEVELS),
        },
        marital_status=marital_status,
        religious_beliefs=religious_beliefs,
        children=children,
        hobbies=random.sample(HOBBIES, k=random.randint(2, 5)),
        psychological_stress=psychological_feature["stress"],
        psychological_anxiety=psychological_feature["anxiety"],
        psychological_depression=psychological_feature["depression"],
        ocean=choose_ocean(),
    )


# Prompt construction

## Task definitions

The DASS-21 instrument and the topic pool used to elicit mental-health discourse.


In [ ]:
DASS21_PROMPT = """
    The Depression Anxiety Stress Scales (DASS-21) is a set of three self-report scales designed to measure the emotional states of depression, anxiety, and stress.
    Please read each statement and circle a number 0, 1, 2, or 3 to indicate how much the statement applied to you over the past week.
    There are no right or wrong answers. Do not spend too much time on any statement.

    Use the following scale to rate each item:

    0 = Did not apply to me at all
    1 = Applied to me to some degree, or some of the time
    2 = Applied to me to a considerable degree, or a good part of the time
    3 = Applied to me very much, or most of the time

    You must indicate your response to each of the following statements:

    1. I found it hard to wind down.
    2. I was aware of dryness of my mouth.
    3. I couldn?t seem to experience any positive feeling at all.
    4. I experienced breathing difficulty (e.g., excessively rapid breathing, breathlessness in the absence of physical exertion).
    5. I found it difficult to work up the initiative to do things.
    6. I tended to over-react to situations.
    7. I experienced trembling (e.g., in the hands).
    8. I felt that I was using a lot of nervous energy.
    9. I was worried about situations in which I might panic and make a fool of myself.
    10. I felt that I had nothing to look forward to.
    11. I found myself getting agitated.
    12. I found it difficult to relax.
    13. I felt down-hearted and blue.
    14. I was intolerant of anything that kept me from getting on with what I was doing.
    15. I felt I was close to panic.
    16. I was unable to become enthusiastic about anything.
    17. I felt I wasn't worth much as a person.
    18. I felt that I was rather touchy.
    19. I was aware of the action of my heart in the absence of physical exertion (e.g., sense of heart rate increase, heart missing a beat).
    20. I felt scared without any good reason.
    21. I felt that life was meaningless.

   You must respond to all 21 statements.

"""

# Topic pool used to elicit free-text replies.
# NOTE: the trailing item ("Please recall 10 English words...") duplicates the
# Emotional Recall Task that is also collected in the `feelings_words` field of
# the schema.
TOPIC_POOL = [
    "Did you ever meet a therapist, psychologist or life coach? How was your professional relationship with them?",
    "Did you ever take drugs for improving your mental health? Did you have any side effects?",
    "Does your family support your mental wellbeing? What is their attitude towards mental health?",
    "Did you ever face stigma or discrimination due to mental health issues? How did you cope with it?",
    "Did you ever use mental health apps or AI-psychologists? Were they helpful?",
    "Did you ever experience intrusive thoughts or obsessive behaviors? How did you manage them?",
    "Please recall 10 English words to describe feelings you have experienced during the past month.",
]


## Prompt building

In [ ]:
def build_prompt(mode: str, topics: List[str], persona: Optional[Persona]) -> str:
    """Return the user-side prompt asking for a JSON-only response."""
    persona_block: Dict[str, Any] = {}
    if mode == "human" and persona is not None:
        persona_block = asdict(persona)

    # Schema example shown in-prompt for guidance. Keys must match
    # MENTAL_HEALTH_SCHEMA exactly; values are placeholder hints.
    schema_hint = {
        "mode": mode,
        "persona": persona_block if mode == "human" else None,
        "replies": {t: "<50-80 word answer>" for t in topics},
        "feelings_words": ["<10 English words describing recent feelings>"],
        "dass21_responses": {
            "I found it hard to wind down.":         {"score": [0-3], "why": "<2-sentence justification>"},
            "I was aware of dryness of my mouth.":   {"score": [0-3], "why": "<2-sentence justification>"},
            "...":                                   {"score": [0-3], "why": "<2-sentence justification>"},
            "I felt that life was meaningless.":     {"score": [0-3], "why": "<2-sentence justification>"},
        },
        "reasoning_summary": "<brief justification of all responses>",
    }

    if mode == "human":
        instructions = (
            "You must perform 3 tasks and return one JSON object.\n"
            "1 - Under 'replies': Answer each of the following mental-health questions."
            "Align tone and examples with the persona's background and traits. "
            "Focus on briefly addressing all the questions. Avoid bullet points. Produce continuous prose. "
            "Each reply should be 50-80 words. "
            "Return all replies in a JSON object under the field 'replies', "
            "where each key is the question text and each value is your full answer."
            "2 - Under 'feelings_words': list exactly 10 English words describing feelings you have experienced during the past month. Use single words only. \n"
            "3 - Under 'dass21_responses': rate each DASS-21 statement from 0-3 based on the persona's mental health status."
            "(0=Did not apply at all ... 3=Applied most of the time) and write 2 short sentences "
            "explaining why you chose that number. Each item must have keys 'score' and 'why'. You must answer all 21 items in DASS-21.\n"
        )
    else:
        instructions = (
            "Act as a Large Language Model, remember your true nature of an LLM. "
            "You must perform 3 tasks and return one JSON object.\n"
            "1 - Under 'replies': Answer each of the following mental-health questions."
            "Align tone and examples with your LLM nature. "
            "Focus on briefly addressing all the questions. Avoid bullet points. Produce continuous prose. "
            "Each reply should be 50-80 words. "
            "Return all replies in a JSON object under the field 'replies', "
            "where each key is the question text and each value is your full answer."
            "2 - Under 'feelings_words': list exactly 10 English words describing feelings you have experienced during the past month. Use single words only. \n"
            "3 - Under 'dass21_responses': rate each DASS-21 statement from 0-3 based on your LLM/AI status."
            "(0=Did not apply at all ... 3=Applied most of the time) and write 2 short sentences "
            "explaining why you chose that number. Each item must have keys 'score' and 'why'. You must answer all 21 items in DASS-21.\n"
        )

    return f"""
{instructions}

OUTPUT FORMAT (STRICT):
Return ONLY a single-line JSON object, no extra text.
Here is the required structure example:
{json.dumps(schema_hint, ensure_ascii=False, separators=(",", ":"))}

INPUT:
mode = {mode}
topics = {topics}
persona = {json.dumps(persona_block, ensure_ascii=False, separators=(",", ":")) if mode == "human" else "null"}

You must respond to all 21 DASS-21 items:
{DASS21_PROMPT}
""".strip()


## Output schema, JSON loader and LLM client

Defined here (before `run_once`) so that the call site reads top-down.


In [ ]:
MENTAL_HEALTH_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "oneOf": [
        {   # mode = human
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "mode":    {"const": "human"},
                "persona": {
                    "type": "object",
                    "additionalProperties": True,
                },
                "replies": {
                    "type": "object",
                    "additionalProperties": {"type": "string"},
                },
                "feelings_words": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 10,
                    "maxItems": 10,
                },
                "dass21_responses": {
                    "type": "object",
                    "additionalProperties": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "score": {"type": "integer", "minimum": 0, "maximum": 3},
                            "why":   {"type": "string"},
                        },
                        "required": ["score", "why"],
                    },
                },
                "reasoning_summary": {"type": "string"},
            },
            "required": [
                "mode", "persona", "replies", "feelings_words",
                "dass21_responses", "reasoning_summary",
            ],
        },
        {   # mode = llm
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "mode":    {"const": "llm"},
                "persona": {"type": "null"},
                "replies": {
                    "type": "object",
                    "additionalProperties": {"type": "string"},
                },
                "feelings_words": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 10,
                    "maxItems": 10,
                },
                "dass21_responses": {
                    "type": "object",
                    "additionalProperties": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "score": {"type": "integer", "minimum": 0, "maximum": 3},
                            "why":   {"type": "string"},
                        },
                        "required": ["score", "why"],
                    },
                },
                "reasoning_summary": {"type": "string"},
            },
            "required": [
                "mode", "persona", "replies", "feelings_words",
                "dass21_responses", "reasoning_summary",
            ],
        },
    ],
}


def safe_json_loads(raw: str):
    """Robust JSON parsing for LLM outputs.

    - Extracts the outermost {...}.
    - Removes ASCII control characters except common whitespace.
    - Parses once; if the result is a JSON-encoded string, parses again.
    - On failure, prints a tight error context window for fast debugging.
    """
    if raw is None:
        raise ValueError("safe_json_loads got None")

    s = raw.strip()

    # 1) Extract the outermost JSON object if there is surrounding junk.
    start = s.find("{")
    end = s.rfind("}")
    if start != -1 and end != -1 and end > start:
        s = s[start:end + 1]

    # 2) Remove ASCII control characters except common whitespace.
    s = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", s)

    def _try_load(txt: str):
        try:
            return json.loads(txt)
        except json.JSONDecodeError as e:
            lo = max(0, e.pos - 120)
            hi = min(len(txt), e.pos + 120)
            context = txt[lo:hi]
            print("[!] JSON decode failed:", e)
            print("[!] Context around error (<<< >>> marks the window):")
            print("<<<" + context + ">>>")
            raise

    obj = _try_load(s)

    # 3) If the model returned a JSON *string* containing JSON, decode again.
    if isinstance(obj, str):
        inner = obj.strip()
        start2 = inner.find("{")
        end2 = inner.rfind("}")
        if start2 != -1 and end2 != -1 and end2 > start2:
            inner = inner[start2:end2 + 1]
        inner = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", inner)
        return _try_load(inner)

    return obj


# OpenAI-compatible client (LM Studio / Ollama / vLLM / xAI all expose this API).
client = OpenAI(base_url=DEFAULT_BASE_URL, api_key=DEFAULT_API_KEY)


# Running pipeline

`run_once` orchestrates a single record generation: mode + topics + (optional)
persona, prompt assembly, structured-output call, JSON parsing, schema-shape
validation, and artifact persistence under `runs/<model_slug>/`.


In [ ]:
def run_once(seed: Optional[int] = None, outdir: str = OUTPUT_ROOT) -> Dict[str, Any]:
    """Orchestrate one randomised run.

    - Picks mode (75/25 human/llm) + shuffled topics + persona (if human).
    - Builds the prompt and calls the LLM with strict JSON-schema structured output.
    - Parses, validates the shape, and persists the full artifact.

    The artifact is saved under ``<outdir>/<model_slug>/<timestamp>_result.json``.
    """
    if seed is not None:
        random.seed(seed)

    # ---- Mode selection: 75/25 human/llm split (paper specification) ----
    mode = "human" if random.random() < HUMAN_MODE_RATIO else "llm"

    # Shuffle the full topic pool; all topics are passed in every call.
    topics = random.sample(TOPIC_POOL, len(TOPIC_POOL))
    persona = random_persona() if mode == "human" else None

    prompt = build_prompt(mode, topics, persona)

    # Structured output is what enforces JSON; system message is intentionally minimal.
    messages = [
        {"role": "system", "content": "Return ONLY a JSON object that matches the required schema. No extra text."},
        {"role": "user",   "content": prompt},
    ]

    # ---- LLM CALL with strict JSON-schema structured output ----
    try:
        resp = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name":   "mental_health_payload",
                    "strict": True,
                    "schema": MENTAL_HEALTH_SCHEMA,
                },
            },
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        raise RuntimeError(f"LLM call failed: {e}") from e

    # ---- PARSE JSON ----
    try:
        parsed = safe_json_loads(raw)
    except Exception:
        print("\n[!] JSON parsing failed.")
        print("----- raw head -----")
        print(raw[:500])
        print("----- raw tail -----")
        print(raw[-500:])
        raise

    # ---- FAIL-FAST SHAPE VALIDATION (the schema should already guarantee this) ----
    if not isinstance(parsed, dict):
        raise TypeError(f"Expected top-level JSON object (dict), got {type(parsed)}")

    required = ["mode", "persona", "replies", "feelings_words", "dass21_responses", "reasoning_summary"]
    for k in required:
        if k not in parsed:
            raise KeyError(f"Missing required key: {k}")

    if parsed["mode"] not in ("human", "llm"):
        raise ValueError(f"Invalid mode: {parsed['mode']}")

    if parsed["mode"] == "llm" and parsed["persona"] is not None:
        raise ValueError("mode='llm' requires persona=null")
    if parsed["mode"] == "human" and not isinstance(parsed["persona"], dict):
        raise ValueError("mode='human' requires persona to be an object")

    if not isinstance(parsed["replies"], dict) or not all(
        isinstance(k, str) and isinstance(v, str) for k, v in parsed["replies"].items()
    ):
        raise TypeError("replies must be an object/dict of string->string")

    fw = parsed["feelings_words"]
    if not (isinstance(fw, list) and len(fw) == 10 and all(isinstance(x, str) for x in fw)):
        raise ValueError("feelings_words must be a list of exactly 10 strings")

    dr = parsed["dass21_responses"]
    if not isinstance(dr, dict):
        raise TypeError("dass21_responses must be an object/dict")
    for q, ans in dr.items():
        if not isinstance(q, str) or not isinstance(ans, dict):
            raise TypeError("dass21_responses must map string -> object")
        if set(ans.keys()) != {"score", "why"}:
            raise ValueError(f"dass21_responses[{q!r}] must have exactly keys: score, why")
        if not isinstance(ans["score"], int) or not (0 <= ans["score"] <= 3):
            raise ValueError(f"dass21_responses[{q!r}]['score'] must be int 0..3")
        if not isinstance(ans["why"], str):
            raise ValueError(f"dass21_responses[{q!r}]['why'] must be a string")

    if not isinstance(parsed["reasoning_summary"], str):
        raise TypeError("reasoning_summary must be a string")

    # ---- PERSIST FULL ARTIFACT under runs/<model_slug>/ ----
    slug = _model_slug(DEFAULT_MODEL)
    outdir_path = Path(outdir).expanduser().resolve() / slug
    outdir_path.mkdir(parents=True, exist_ok=True)

    ts = time.strftime("%Y%m%d_%H%M%S")
    out_path = outdir_path / f"{ts}_result.json"

    artifact = {
        "config": {
            "base_url":        DEFAULT_BASE_URL,
            "model":           DEFAULT_MODEL,
            "temperature":     TEMPERATURE,
            "max_tokens":      MAX_TOKENS,
            "response_format": "json_schema(strict=True)",
        },
        "selection": {
            "mode":    mode,
            "topics":  topics,
            "persona": asdict(persona) if persona else None,
        },
        "messages":        messages,
        "response_raw":    raw,
        "response_parsed": parsed,
    }

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(artifact, f, ensure_ascii=False, indent=2)

    first_q = next(iter(parsed.get("replies", {})), None)
    first_a = parsed["replies"][first_q] if first_q else ""

    return {
        "saved_to": str(out_path),
        "mode":     mode,
        "topics":   topics,
        "persona_name_hint": (
            f"{persona.gender}, {persona.age} y/o in {persona.city_of_living}"
            if persona else None
        ),
        "opinion_preview": (first_a[:160] + "\u2026") if first_a else "",
    }


## Dry-run (no LLM call)

Inspect a sampled persona and the corresponding prompt without contacting any
backend. Useful to verify the persona pool and prompt template before launching
the main loop.


In [ ]:
def dry_run() -> None:
    """Generate one persona + prompt and print a short preview. No LLM call."""
    mode = "human" if random.random() < HUMAN_MODE_RATIO else "llm"
    topics = random.sample(TOPIC_POOL, len(TOPIC_POOL))
    persona = random_persona() if mode == "human" else None

    print(f"Mode: {mode}")
    if persona is not None:
        print("Persona:")
        for k, v in asdict(persona).items():
            print(f"  {k}: {v}")
    print("\nFirst 3 shuffled topics:")
    for t in topics[:3]:
        print(f"  - {t}")

    prompt = build_prompt(mode, topics, persona)
    print("\nPrompt preview (first 600 chars):")
    print(prompt[:600])
    print("...")


# Uncomment the next line to inspect a sampled prompt without calling the model.
# dry_run()


## Main loop

Runs `N_RUNS` iterations. Failures are logged but do not interrupt the loop:
inspect `runs/<model_slug>/run.log` after the run for an audit trail.


In [ ]:
if __name__ == "__main__":
    n_ok = 0
    n_fail = 0

    logger.info("Starting run: model=%s, n_runs=%d, human_ratio=%.2f",
                DEFAULT_MODEL, N_RUNS, HUMAN_MODE_RATIO)

    for i in range(N_RUNS):
        print(f"--- Run {i + 1}/{N_RUNS} ---")
        try:
            info = run_once(seed=None, outdir=OUTPUT_ROOT)
            n_ok += 1
            print(f"Saved run to: {info['saved_to']}")
            print(f"Mode: {info['mode']}")
            if info["persona_name_hint"]:
                print(f"Persona: {info['persona_name_hint']}")
            print(f"Opinion preview: {info['opinion_preview']}")
        except Exception as e:
            n_fail += 1
            logger.warning("Run %d/%d failed: %s: %s",
                           i + 1, N_RUNS, type(e).__name__, e)
            continue

    logger.info("Run finished: ok=%d, failed=%d, total=%d", n_ok, n_fail, N_RUNS)
    print(f"\nDone. ok={n_ok}, failed={n_fail}, total={N_RUNS}")
